In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from numba import njit

random.seed(13)

# %%
def lagm_cov_est(ret, m):
    
    N = len(ret)
    Ybar = np.mean(ret)
    
    cov = np.sum((ret[:N-m] - Ybar) * (ret[m:] - Ybar)) / (N - m)
    return cov

# %%
def estimate_moments(ret):
    n = len(ret)
    Ybar = np.mean(ret)
    mean = Ybar
    var = np.var(ret, ddof = 1)
    lag1 = lagm_cov_est(ret, 1) 
    lag2 = lagm_cov_est(ret, 2)

    y2 = ret[:-1]**2
    y2d = y2 - np.mean(y2)
    yn1d = ret[1:] - np.mean(ret[1:])
    cross = np.mean(y2d * yn1d)
    
    return np.array([mean, var, lag1, lag2, cross])


def moment_conditions(ret, params):
    mu, theta, kappa, sigma_V, rho = params
    N = len(ret)
    Ybar = np.mean(ret)
    Y2bar = np.mean(ret ** 2)
    
    # Moments per obs
    z = np.zeros((N, 5))
    z[:, 0] = ret - Ybar                                    # mean
    z[:, 1] = (ret - Ybar) ** 2 - np.var(ret, ddof = 1)     # var
    cov1 = lagm_cov_est(ret, 1)
    z[:-1, 2] = (ret[:-1] - Ybar) * (ret[1:] - Ybar) - cov1     # lag1

    cov2 = lagm_cov_est(ret, 2)
    z[:-2, 3] = (ret[:-2] - Ybar) * (ret[2:] - Ybar) - cov2     # lag2

    y2    = ret[:-1]**2
    y2bar = np.mean(y2)
    cross = np.mean((y2 - y2bar) * (ret[1:] - np.mean(ret[1:])))
    z[:-1, 4] = (y2 - y2bar) * (ret[1:] - Ybar) - cross         # cross
    return z

def newey_west(z, lag_max):
    
    N, k = z.shape
    Sigma = (z.T @ z) / N # lag-0 term
    
    for lag in range(1, lag_max + 1):
        w = 1 - lag / (lag_max + 1)
        Gamma = (z[lag:].T @ z[:-lag]) / N
        Sigma += w * (Gamma + Gamma.T)
        
    return Sigma

def estimate_kappa(ret, h , lag_M = 10):
    N = len(ret)
    
    lag1 = lagm_cov_est(ret, 1)
    
    kappa_sum = 0
    
    for m in range(2, lag_M + 1):
        lagm = lagm_cov_est(ret, m)
        
        if lagm == 0 or np.sign(lag1) != np.sign(lagm):
            continue
        
        ratio = lag1 / lagm
        if ratio <= 0 or not np.isfinite(np.log(ratio)):
            continue
        
        k_est = np.log(ratio) / ((m-1) * h)
        if not np.isfinite(k_est) or k_est <= 0:
            continue
        kappa_sum += k_est
        
    kappa = kappa_sum / (lag_M - 1)
    return kappa

# %%
def h_tilde(kappa, h):
    "h_tidle = (1 - exp(-kappa * h)) / kappa"
    return (1 - np.exp(-kappa * h)) / kappa

# %%
@njit
def heston_returns(V0, N, mu, theta, kappa, sigma_V, rho, h, n_sub):
    
    dt = h / n_sub
    sqrt_dt = dt ** 0.5
    c = (1 - rho ** 2) ** 0.5 
    
    returns = np.empty(N)
  
    Vt = V0
    log_ret = 0
    
    for n in range(N):
        log_ret = 0
        for _ in range(n_sub):
            z1 = np.random.normal(0, 1)
            z2 = np.random.normal(0, 1)
            
            dB1 = sqrt_dt * z1
            dB2 = sqrt_dt * (rho * z1 + c * z2) 
            
            Vt_pos = Vt if Vt > 0 else 0 # ensure variance is non-negative for the price update
            
            log_ret += (mu - 0.5 * Vt) * dt + Vt_pos ** 0.5 * dB1 # log return increment
            
            Vt = Vt_pos + kappa * (theta - Vt_pos)* dt + sigma_V * Vt_pos ** 0.5 * dB2  # variance process
            
        returns[n] = log_ret
        
    return returns 

def estimate_heston_from_returns(ret, h):
    N = len(ret)
    mean_e, var_e, lag1_e, lag2_e, cross_e = estimate_moments(ret)
    
    kappa = estimate_kappa(ret, h)
    print( f"  κ̂    = {kappa:.4f}")
    
    ht = h_tilde(kappa, h)
    
    ek = np.exp(-kappa * h)
    
    dh = h * ek - ht
    
    theta = var_e / h  - (2* (h-ht) / (h * kappa * ht ** 2)) * lag1_e 
    print(f"  θ̂    = {theta:.4f}")
    
    mu = mean_e / h + theta / 2
    print(f"  μ̂    = {mu:.4f}")
    
    num   = (4*kappa*mean_e
             + (8*dh/(theta*ht**3))*lag1_e
             - 2*kappa*cross_e/lag1_e)
    den   = (theta*ht**2/(2*lag1_e) - dh/(kappa*ht))

    sigma_sq = num / den
    sigma_V = sigma_sq**0.5
    print(f"  σ̂_V  = {sigma_V:.6f}")

    rho = np.clip(sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
                  -0.999, 0.999)
    print(f"  ρ̂    = {rho:.6f}")

    return {'mu': mu, 'theta': theta, 'kappa': kappa,
            'sigma_V': sigma_V, 'rho': rho}

# %%
def estimate_from_moments(samp, h, extra_lags=None):
    """
    samp       : (mean, var, lag1, lag2, cross)
    extra_lags : array of lagm_cov_est(ret, m) for m=2..lag_M, or None
    """
    mean_e, var_e, lag1_e, lag2_e, cross_e = samp

    if extra_lags is not None and len(extra_lags) > 0:
        kappa_sum = 0
        count     = 0
        for m, lagm in enumerate(extra_lags, start=2):
            if lagm == 0 or np.sign(lag1_e) != np.sign(lagm):
                continue
            ratio = lag1_e / lagm
            if ratio <= 0 or not np.isfinite(np.log(ratio)):
                continue
            k_est = np.log(ratio) / ((m - 1) * h)
            if not np.isfinite(k_est) or k_est <= 0:
                continue
            kappa_sum += k_est
            count     += 1
        if count == 0:
            raise ValueError("No valid lag ratios.")
        kappa = kappa_sum / count
    else:
        # fallback: single ratio
        if lag1_e == 0 or np.sign(lag1_e) != np.sign(lag2_e):
            raise ValueError("Invalid lag ratio.")
        ratio = lag2_e / lag1_e
        if not (0 < ratio < 1):
            raise ValueError(f"Ratio {ratio:.4f} not in (0,1).")
        kappa = -np.log(ratio) / h

    kappa   = -np.log(ratio) / h
    ht      = (1 - np.exp(-kappa*h)) / kappa
    ek      = np.exp(-kappa*h)
    dh      = h*ek - ht
    theta   = var_e/h - (2*(h-ht)/(h*kappa*ht**2))*lag1_e

    if not np.isfinite(theta) or theta <= 0:
        raise ValueError(f"Invalid theta = {theta:.4e}")

    mu      = mean_e/h + theta/2
    num     = (4*kappa*mean_e
               + (8*dh/(theta*ht**3))*lag1_e
               - 2*kappa*cross_e/lag1_e)
    den     = theta*ht**2/(2*lag1_e) - dh/(kappa*ht)

    if abs(den) < 1e-14:
        raise ValueError("Denominator zero.")

    sigma_sq = num / den

    # This allows the Jacobian to compute a finite derivative
    # even when the perturbed moments push sigma_sq slightly negative
    min_sigma_sq = 1e-10
    if sigma_sq <= 0:
        sigma_sq = min_sigma_sq   # boundary value — derivative still defined

    sigma_V = sigma_sq**0.5
    rho     = np.clip(
        sigma_V/(4*kappa) - (2/(theta*sigma_V*ht**2))*lag1_e,
        -0.999, 0.999
    )

    return {'kappa': kappa, 'theta': theta, 'sigma_V': sigma_V,
            'mu': mu, 'rho': rho}


In [3]:
def experiment_v0_burnin(TRUE, h, n_sub, n_rep=100, N=int(400_000)):
    burnin  = int(5_000)
    configs = {
        'A: V0=θ (stationary)'   : (TRUE['theta'],       0),
        'B: V0=4θ (no burnin)'   : (4*TRUE['theta'],     0),
        'C: V0=0.1θ (no burnin)' : (0.1*TRUE['theta'],   0),
        'D: V0=4θ (burnin)'      : (4*TRUE['theta'],     burnin),
        'E: V0=0.1θ (burnin)'    : (0.1*TRUE['theta'],   burnin),  # NEW
    }
    results = {name: {key: [] for key in ['kappa','theta','sigma_V','mu','rho']}
               for name in configs}
    failed  = {name: 0 for name in configs}

    for name, (V0, burn) in configs.items():
        print(f"\nRunning config: {name}")
        for r in range(n_rep):
            ret_full = heston_returns(V0, N + burn,
                                      TRUE['mu'], TRUE['theta'], TRUE['kappa'],
                                      TRUE['sigma_V'], TRUE['rho'], h, n_sub)
            ret = ret_full[burn:]
            try:
                est = estimate_heston_from_returns(ret, h)
                if not np.isfinite(est['kappa']) or est['kappa'] <= 0:
                    raise ValueError(f"Invalid kappa = {est['kappa']:.4e}")
                for key in results[name]:
                    results[name][key].append(est[key])
            except Exception as ex:
                failed[name] += 1

        print(f"  Failed: {failed[name]}/{n_rep}")

    # Print bias comparison
    print(f"\n{'Config':<25s}  {'κ bias':>10s}  {'θ bias':>10s}  "
          f"{'σ_V bias':>10s}  {'μ bias':>10s}  {'ρ bias':>10s}  "
          f"{'Failed':>8s}")
    print("-" * 85)
    for name in configs:
        vals = results[name]
        biases = {key: np.nanmean(vals[key]) - TRUE[key]
                  for key in ['kappa','theta','sigma_V','mu','rho']}
        print(f"  {name:<23s}  "
              f"{biases['kappa']:10.4f}  {biases['theta']:10.4f}  "
              f"{biases['sigma_V']:10.4f}  {biases['mu']:10.4f}  "
              f"{biases['rho']:10.4f}  {failed[name]:>8d}")

    return results, failed

In [4]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.1, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)

# Start with n_rep=10 to test — takes ~2 minutes with numba
burnin = experiment_v0_burnin(TRUE, h, n_sub, n_rep = 100, N=int(100_000))


Running config: A: V0=θ (stationary)
  κ̂    = 0.1004
  θ̂    = 0.2494
  μ̂    = 0.1222
  σ̂_V  = 0.092813
  ρ̂    = -0.809031
  κ̂    = 0.0758
  θ̂    = 0.2475
  μ̂    = 0.1258
  σ̂_V  = 0.087829
  ρ̂    = -0.689174
  κ̂    = 0.1415
  θ̂    = 0.2492
  μ̂    = 0.1234
  σ̂_V  = 0.112490
  ρ̂    = -0.808007
  κ̂    = 0.1298
  θ̂    = 0.2475
  μ̂    = 0.1259
  σ̂_V  = 0.108687
  ρ̂    = -0.738326
  κ̂    = 0.1198
  θ̂    = 0.2498
  μ̂    = 0.1263
  σ̂_V  = 0.102437
  ρ̂    = -0.714280
  κ̂    = 0.1165
  θ̂    = 0.2477
  μ̂    = 0.1236
  σ̂_V  = 0.113365
  ρ̂    = -0.671878
  κ̂    = 0.1068
  θ̂    = 0.2488
  μ̂    = 0.1254
  σ̂_V  = 0.086822
  ρ̂    = -0.810885
  κ̂    = 0.0865
  θ̂    = 0.2528
  μ̂    = 0.1263
  σ̂_V  = 0.098548
  ρ̂    = -0.589493
  κ̂    = 0.1286
  θ̂    = 0.2480
  μ̂    = 0.1232
  σ̂_V  = 0.132577
  ρ̂    = -0.578005
  κ̂    = 0.1003
  θ̂    = 0.2449
  μ̂    = 0.1260
  σ̂_V  = 0.092868
  ρ̂    = -0.825329
  κ̂    = 0.0665
  θ̂    = 0.2462
  μ̂    = 0.1253
  σ̂_V  = 0

In [5]:
# ── Run ───────────────────────────────────────────────────────────────────────
TRUE  = dict(mu=0.125, kappa=0.01, theta=0.25, sigma_V=0.1, rho=-0.7)
V0    = TRUE['theta']
h     = 1.0
n_sub = int(20)

# Start with n_rep=10 to test — takes ~2 minutes with numba
burnin2 = experiment_v0_burnin(TRUE, h, n_sub, n_rep = 100, N=int(100_000))


Running config: A: V0=θ (stationary)
  κ̂    = 0.0071
  θ̂    = 0.2510
  μ̂    = 0.1255
  σ̂_V  = 0.086510
  ρ̂    = -0.636682
  κ̂    = 0.0098
  θ̂    = 0.2671
  μ̂    = 0.1252
  σ̂_V  = 0.102905
  ρ̂    = -0.790242
  κ̂    = 0.0071
  θ̂    = 0.2572
  μ̂    = 0.1257
  σ̂_V  = 0.087433
  ρ̂    = -0.748884
  κ̂    = 0.0050
  θ̂    = 0.2504
  μ̂    = 0.1249
  σ̂_V  = 0.077119
  ρ̂    = -0.448826
  κ̂    = 0.0063
  θ̂    = 0.2448
  μ̂    = 0.1255
  σ̂_V  = 0.077151
  ρ̂    = -0.753739
  κ̂    = 0.0185
  θ̂    = 0.2209
  μ̂    = 0.1268
  σ̂_V  = 0.129140
  ρ̂    = -0.448991
  κ̂    = 0.0286
  θ̂    = 0.2363
  μ̂    = 0.1247
  σ̂_V  = 0.158152
  ρ̂    = -0.630193
  κ̂    = 0.0338
  θ̂    = 0.2427
  μ̂    = 0.1243
  σ̂_V  = 0.170073
  ρ̂    = -0.535747
  κ̂    = 0.0273
  θ̂    = 0.2436
  μ̂    = 0.1250
  σ̂_V  = 0.176896
  ρ̂    = -0.434527
  κ̂    = 0.0243
  θ̂    = 0.2661
  μ̂    = 0.1233
  σ̂_V  = 0.162198
  ρ̂    = -0.556923
  κ̂    = 0.0119
  θ̂    = 0.2494
  μ̂    = 0.1249
  σ̂_V  = 0

C:\Users\karis\AppData\Local\Temp\ipykernel_51532\3548403523.py:96: RuntimeWarning: invalid value encountered in scalar divide
  return (1 - np.exp(-kappa * h)) / kappa


  κ̂    = 0.0047
  θ̂    = 0.2812
  μ̂    = 0.1237
  σ̂_V  = 0.069716
  ρ̂    = -0.748775
  κ̂    = 0.0146
  θ̂    = 0.2470
  μ̂    = 0.1247
  σ̂_V  = 0.120251
  ρ̂    = -0.514817
  κ̂    = 0.0026
  θ̂    = 0.2508
  μ̂    = 0.1276
  σ̂_V  = 0.050351
  ρ̂    = -0.907850
  κ̂    = 0.0260
  θ̂    = 0.2489
  μ̂    = 0.1242
  σ̂_V  = 0.167764
  ρ̂    = -0.506120
  κ̂    = 0.0226
  θ̂    = 0.2615
  μ̂    = 0.1230
  σ̂_V  = 0.155775
  ρ̂    = -0.506151
  κ̂    = 0.0087
  θ̂    = 0.2463
  μ̂    = 0.1273
  σ̂_V  = 0.091731
  ρ̂    = -0.709929
  κ̂    = 0.0118
  θ̂    = 0.2589
  μ̂    = 0.1250
  σ̂_V  = 0.109120
  ρ̂    = -0.659915
  κ̂    = 0.0147
  θ̂    = 0.2397
  μ̂    = 0.1250
  σ̂_V  = 0.112075
  ρ̂    = -0.881572
  κ̂    = 0.0012
  θ̂    = 0.2568
  μ̂    = 0.1273
  σ̂_V  = 0.030076
  ρ̂    = -0.999000
  κ̂    = 0.0163
  θ̂    = 0.2532
  μ̂    = 0.1227
  σ̂_V  = 0.141449
  ρ̂    = -0.510400
  κ̂    = 0.0058
  θ̂    = 0.2608
  μ̂    = 0.1233
  σ̂_V  = 0.073920
  ρ̂    = -0.999000
  κ̂    = 